In [ ]:
import pandas as pd
df = pd.read_csv('C:/Users/tyler/OneDrive/Documents/GitHub/work_to_show/Sports/Datasets SA Class/SBR001-715.csv')

print(df.columns.tolist())

In [ ]:
# descriptive stats only numeric columns 
years = df.columns[1:]  # columns to the right

for item in df['Item'].dropna().unique():
    values = pd.to_numeric(
        df.loc[df['Item'] == item, years].iloc[0],
        errors='coerce'
    )

    # skip if ALL values are NaN
    if values.notna().sum() == 0:
        continue

    print(f"\n=== {item} ===")
    print(values.describe())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

# -------------------------------
# 1. DESCRIPTIVE STATS PER ITEM
# -------------------------------
years = df.columns[1:]  # all year columns

for item in df['Item'].dropna().unique():

    values = pd.to_numeric(
        df.loc[df['Item'] == item, years].iloc[0],
        errors='coerce'
    )

    # Skip items with no numeric data
    if values.notna().sum() == 0:
        continue

    print(f"\n=== {item} ===")
    print(values.describe())

# -------------------------------
# 2. FEATURE / TARGET SETUP
# -------------------------------
VALUES = [
    '2013.00','2014.00','2015.00','2016.00','2017.00',
    '2018.00','2019.00','2020.00','2021.00'
]
TARGET_COL = '2022.00'

# Ensure numeric
df[VALUES + [TARGET_COL]] = df[VALUES + [TARGET_COL]].apply(
    pd.to_numeric, errors='coerce'
).fillna(0)

# -------------------------------
# 3. TRAIN / TEST SPLIT
# -------------------------------
np.random.seed(42)
df['r'] = np.random.rand(len(df))

train = df[df['r'] <= 0.6]
test  = df[df['r'] > 0.6]

X = train[VALUES]
y = train[TARGET_COL]

# -------------------------------
# 4. RANDOM FOREST REGRESSION
# -------------------------------
rf = RandomForestRegressor(
    n_estimators=500,
    random_state=17
)

rf.fit(X, y)

# -------------------------------
# 5. FEATURE IMPORTANCE
# -------------------------------
importances = rf.feature_importances_
sorted_indices = np.argsort(importances)[::-1]

print("\n=== Feature Importance (Predicting 2022) ===")
for i in sorted_indices:
    print(f"{VALUES[i]:<10} {importances[i]:.4f}")

# -------------------------------
# 6. FEATURE IMPORTANCE PLOT
# -------------------------------
plt.figure(figsize=(8, 4))
plt.title('Feature Importance (2013–2021 → 2022)')
plt.bar(range(len(VALUES)), importances[sorted_indices])
plt.xticks(
    range(len(VALUES)),
    np.array(VALUES)[sorted_indices],
    rotation=45
)
plt.tight_layout()
plt.show()


In [ ]:
# what is the soical media followers over time?
social_media = df.loc[
    df['Item'].isin([
        'Follow Minor League Baseball on Facebook - Total all Followers (add 000) ',
        'Follow Minor League Baseball on X (formerly Twitter) - Total all Followers (add 000)'
    ])
]

# graph social media (run chart)

# identify year columns 
year_cols = df.columns[1:]  # everything after 'Item'
social_media = social_media[['Item'] + list(year_cols)]

# convert to numeric
social_media[year_cols] = social_media[year_cols].apply(
    pd.to_numeric, errors='coerce'
)

# reshape (wide-> long)
social_media_long = social_media.melt(
    id_vars='Item',
    var_name='Year',
    value_name='Followers'
)

# plot the chart 
import matplotlib.pyplot as plt

plt.figure(figsize=(9,5))

for platform in social_media_long['Item'].unique():
    subset = social_media_long[social_media_long['Item'] == platform]
    plt.plot(
        subset['Year'],
        subset['Followers'],
        marker='o',
        label=platform
    )

plt.title('MiLB Social Media Followers Over Time')
plt.xlabel('Year')
plt.ylabel('Followers (000s)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

'''
What to focus on: 
social media. stays pretty consitant from 2016-2018
2019 had the worst social media 
'''


In [ ]:
# what is the attendance over time?
attendance = df.loc[
    df['Item'].isin([
        'Attended at Least One Minor League Baseball Game',
        'Attended 4+ Minor League Baseball Games'
    ])
]

# graph social media (run chart)

# identify year columns 
year_cols = df.columns[1:]  # everything after 'Item'
attendance = attendance[['Item'] + list(year_cols)]

# convert to numeric
attendance[year_cols] = attendance[year_cols].apply(
    pd.to_numeric, errors='coerce'
)

# reshape (wide-> long)
attendance_long = attendance.melt(
    id_vars='Item',
    var_name='Year',
    value_name='Followers'
)

# plot the chart 
import matplotlib.pyplot as plt

plt.figure(figsize=(9,5))

for platform in attendance_long['Item'].unique():
    subset = attendance_long[attendance_long['Item'] == platform]
    plt.plot(
        subset['Year'],
        subset['Followers'],
        marker='o',
        label=platform
    )

plt.title('MiLB Social Media Followers Over Time')
plt.xlabel('Year')
plt.ylabel('Attendance (000s)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

'''
What to focus on: 
why is some years have more 4+ games attendanded
'''


In [ ]:
df.reset_index()
# Drop empty columns if necessary
df2 = df.dropna(axis=1, how='all')

# Melt the dataframe to long format
long_df = df2.melt(id_vars=['Item'], var_name='Year', value_name='Value')

# Convert Year and Value to numeric
long_df['Year'] = pd.to_numeric(long_df['Year'], errors='coerce')
long_df['Value'] = pd.to_numeric(long_df['Value'], errors='coerce')

df3=long_df
print(df3.columns.tolist())

In [52]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# -------------------------------
# 1. Load and clean data
# -------------------------------
df = pd.read_csv('C:/Users/tyler/OneDrive/Documents/GitHub/work_to_show/Sports/Datasets SA Class/SBR001-715.csv')

# Drop empty columns
df2 = df.dropna(axis=1, how='all')

# Melt to long format
long_df = df2.melt(id_vars=['Item'], var_name='Year', value_name='Value')

# Convert to numeric
long_df['Year'] = pd.to_numeric(long_df['Year'], errors='coerce')
long_df['Value'] = pd.to_numeric(long_df['Value'], errors='coerce')

df3 = long_df

# -------------------------------
# 2. Select social media items
# -------------------------------
social_media_items = df3.loc[
    df3['Item'].str.contains('Follow Minor League Baseball on', na=False)
]['Item'].unique()

predict_year = 2026
results = {}

# -------------------------------
# 3. Loop through social media items
# -------------------------------
for item in social_media_items:
    data = df3[df3['Item'] == item].dropna(subset=['Year','Value'])
    
    if data.empty:
        continue

    # Optional: log-transform to account for exponential growth
    data['Value_log'] = np.log1p(data['Value'])
    
    # Use both Year and previous trend as features
    X = sm.add_constant(data['Year'])
    y = data['Value_log']  # log-transformed target
    
    # Fit OLS model
    model = sm.OLS(y, X).fit()
    
    # Predict 2026 (and back-transform)
    X_pred = pd.DataFrame({'const':1, 'Year':[predict_year]})
    pred_2026 = np.expm1(model.predict(X_pred))[0]  # inverse log transform
    results[item] = int(pred_2026)
    print(f"Predicted {item} in {predict_year}: {int(pred_2026)}")

    # -------------------------------
    # Plot with prediction interval
    # -------------------------------
    st, data_table, ss2 = sm.stats.outliers_influence.summary_table(model, alpha=0.05)
    fittedvalues = data_table[:,2]
    predict_ci_low, predict_ci_upp = data_table[:,6:8].T
    
    plt.figure(figsize=(6,4))
    plt.plot(data['Year'], np.expm1(y), 'o', label='Actual')
    plt.plot(data['Year'], np.expm1(fittedvalues), '-', lw=2, label='Fit')
    plt.plot(data['Year'], np.expm1(predict_ci_low), 'r--', lw=1, label='95% PI')
    plt.plot(data['Year'], np.expm1(predict_ci_upp), 'r--', lw=1)
    plt.scatter(predict_year, pred_2026, color='purple', zorder=5, label=f'Pred {predict_year}')
    plt.title(f"{item} Prediction (Log-Trend Adjusted)")
    plt.xlabel('Year')
    plt.ylabel('Followers (000s)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


Predicted Follow Minor League Baseball on Facebook - Total all Followers (add 000)  in 2026: 8980


AttributeError: module 'statsmodels.stats.api' has no attribute 'outliers_influence'